# Assignment 2: NLP — PDF Question-Answering Tool

**Name:** Md. JADID HOSSAIN SANIM
**ID:** 1000129287

---



---
# Part 1: Setup and Installation

Install the required libraries and verify the runtime environment.


In [ ]:
# Install all required libraries
!pip install -q pymupdf transformers sentencepiece tiktoken scikit-learn gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 79.5 MB/s eta 0:00:00


In [ ]:
import re
import math
import textwrap

import numpy as np
import pandas as pd
import tiktoken
import torch
import pymupdf

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

print("Libraries imported successfully")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Libraries imported successfully
GPU available: True
Device: Tesla T4


---
# Part 2: Upload the PDF

The tool answers questions from any pdf that we give it to them.

In [ ]:
# Upload the PDF the tool should answer questions from
from google.colab import files

print("Choose a PDF file to upload:")
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]

print(f"\nDocument to query: {PDF_PATH}")

Choose a PDF file to upload:


Saving A-History-of-Bangladesh.pdf to A-History-of-Bangladesh.pdf

Document to query: A-History-of-Bangladesh.pdf


---
# Part 3: PDF Text Extraction and Cleaning




In [ ]:
def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """
    Extract the text of every page with PyMuPDF.
    Returns a list of {"page": page_number, "text": page_text}.
    """
    doc = pymupdf.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc):
        pages.append({"page": page_num + 1, "text": page.get_text("text")})
    doc.close()
    return pages


pages = extract_text_from_pdf(PDF_PATH)
print(f"Extracted {len(pages)} page(s)")

print("\n--- raw text of page 1 (first 400 characters) ---")
print(pages[0]["text"][:400])

Extracted 375 page(s)

--- raw text of page 1 (first 400 characters) ---



In [ ]:
NOISE_RE = re.compile(r"[(_$%&]{5,}")  # 5+ consecutive noise characters = garbage

def clean_text(text: str) -> str:
    """
    Cleaning steps:
    1. remove long noise runs like (_$$$$%%%%&&&&((((( — some PDFs
       are full of these garbage characters
    2. remove standalone page numbers
    3. normalize whitespace and punctuation spacing
    """
    text = NOISE_RE.sub(" ", text)                 # noise runs
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)  # standalone page numbers
    text = re.sub(r"[ \t]+", " ", text)           # collapse spaces / tabs
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)   # " ." -> "."
    text = re.sub(r"\n{3,}", "\n\n", text)        # collapse blank lines
    return text.strip()


for p in pages:
    p["text"] = clean_text(p["text"])

print("--- page 1 after cleaning (first 400 characters) ---")
print(pages[0]["text"][:400])

noise_runs = NOISE_RE.findall("\n".join(pg["text"] for pg in extract_text_from_pdf(PDF_PATH)))
print(f"\nNoise runs found in the raw document: {len(noise_runs)}")
if noise_runs:
    print("Example:", noise_runs[0][:60] + "...")

--- page 1 after cleaning (first 400 characters) ---


Noise runs found in the raw document: 0


In [ ]:
# Document statistics
enc = tiktoken.get_encoding("cl100k_base")
full_text = "\n".join(p["text"] for p in pages)
total_tokens = len(enc.encode(full_text))

print(f"Pages: {len(pages)} | Words: {len(full_text.split()):,} | Tokens (approx): {total_tokens:,}")
print()
print("The QA model (RoBERTa) accepts at most 512 tokens per input,")
print("so the document is split into chunks that fit its context window.")

Pages: 375 | Words: 110,689 | Tokens (approx): 186,348

The QA model (RoBERTa) accepts at most 512 tokens per input,
so the document is split into chunks that fit its context window.


---
# Part 4: Chunking

The QA model can only read 512 tokens at a time, so the cleaned text is split
into paragraph-aware chunks. Paragraphs are packed together until a chunk
is full. each chunk keeps its page number for source citation.


In [ ]:
def make_chunks(pages: list[dict], max_chars: int = 900, min_chars: int = 150) -> list[dict]:
    """
    Pack the paragraphs of each page into chunks of at most `max_chars`.
    Very short trailing chunks are merged into the previous one.
    Every chunk remembers the page it came from.
    """
    chunks: list[dict] = []

    for p in pages:
        paragraphs = [t.strip() for t in re.split(r"\n\s*\n", p["text"]) if t.strip()]

        buf = ""
        for para in paragraphs:
            if buf and len(buf) + len(para) + 1 > max_chars:
                chunks.append({"id": len(chunks), "page": p["page"], "text": buf})
                buf = ""
            buf = f"{buf} {para}".strip() if buf else para
            while len(buf) > max_chars:          # single huge paragraph -> hard split
                cut = buf.rfind(" ", 0, max_chars)
                cut = cut if cut > 0 else max_chars
                chunks.append({"id": len(chunks), "page": p["page"], "text": buf[:cut]})
                buf = buf[cut:].strip()
        if buf:
            chunks.append({"id": len(chunks), "page": p["page"], "text": buf})

    merged: list[dict] = []
    for c in chunks:
        if merged and len(c["text"]) < min_chars:
            merged[-1]["text"] += " " + c["text"]
        else:
            merged.append(c)
    for i, c in enumerate(merged):
        c["id"] = i
    return merged


chunks = make_chunks(pages)
print(f"Created {len(chunks)} chunks:")
for c in chunks:
    preview = c["text"][:60].replace("\n", " ")
    print(f"  chunk {c['id']:>2} | page {c['page']} | ~{len(enc.encode(c['text'])):>3} tokens | {preview}...")

Created 889 chunks:
  chunk  0 | page 2 | ~193 tokens | A HISTORY OF BANGLADESH Bangladesh is a new name for an old ...
  chunk  1 | page 2 | ~113 tokens | and its resilient and inventive people. willem van schendel ...
  chunk  2 | page 5 | ~203 tokens | CAMBRIDGE UNIVERSITY PRESS Cambridge, New York, Melbourne, M...
  chunk  3 | page 5 | ~ 85 tokens | responsibility for the persistence or accuracy of URLs for e...
  chunk  4 | page 6 | ~153 tokens | Contents List of plates page vii List of maps and ﬁgures xiv...
  chunk  5 | page 7 | ~140 tokens | 12. Pakistan falls apart 13. East Pakistani livelihoods 14. ...
  chunk  6 | page 8 | ~240 tokens | Plates* Part i Aerial view of the Sundarbans wetlands. Court...
  chunk  7 | page 8 | ~142 tokens | York. 2.5 Ruins of the northern gateway to the fort of Gaur,...
  chunk  8 | page 9 | ~294 tokens | 3.2 Scene from the legend of Gazi Pir, showing the pir ridin...
  chunk  9 | page 9 | ~263 tokens | Twenty-four enlightened ascetics. From: Annu

---
# Part 5: Retrieval — finding the right chunk with TF-IDF



In [ ]:
class TfidfRetriever:
    """Classic IR: TF-IDF chunk vectors + cosine similarity search."""

    def __init__(self, chunks: list[dict]):
        self.chunks = chunks
        self.vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=True)
        self.matrix = self.vectorizer.fit_transform([c["text"] for c in chunks])

    def top_k(self, question: str, k: int = 3) -> list:
        """Return the [(chunk, similarity)] of the k most relevant chunks."""
        sims = cosine_similarity(self.vectorizer.transform([question]), self.matrix)[0]
        best = np.argsort(sims)[::-1][:k]
        return [(self.chunks[i], float(sims[i])) for i in best if sims[i] > 0]


retriever = TfidfRetriever(chunks)
print(f"TF-IDF index built over {len(chunks)} chunks")

demo_question = "What is the official language of Bangladesh?"
print(f"\nQuery: {demo_question}\n")
for chunk, sim in retriever.top_k(demo_question, k=3):
    print(f"  chunk {chunk['id']:>2} (page {chunk['page']})  cosine similarity = {sim:.3f}")
    print("  " + chunk["text"][:140].replace("\n", " ") + "...\n")

TF-IDF index built over 889 chunks

Query: What is the official language of Bangladesh?

  chunk 116 (page 59)  cosine similarity = 0.238
  single category: that of ethnic minorities facing the dominant ethnic identity in the country, Islamic Bengali. the language frontier The st...

  chunk 293 (page 137)  cosine similarity = 0.215
  Constituent Assembly tabled an amendment to allow the Bengali language to be used in the Assembly alongside Urdu. He was sharply rebutted by...

  chunk 298 (page 138)  cosine similarity = 0.213
  suspect. When students in East Pakistan came to know about the plan to make Urdu the national language, they held meetings and demonstration...



---
# Part 6: The QA Model — RoBERTa trained on SQuAD 2.0


The model reads a (question, context) pair and produces two scores per token:
how likely it **starts** the answer and how likely it **ends** the answer.
The best (start, end) pair forms the answer span.




In [ ]:
QA_MODEL_NAME = "deepset/roberta-base-squad2"

tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
print(f"QA model '{QA_MODEL_NAME}' loaded on {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

QA model 'deepset/roberta-base-squad2' loaded on cuda


In [ ]:
@torch.no_grad()
def extract_answer(question: str, context: str, n_best: int = 20, max_span_len: int = 30) -> dict:
    """
    Run the QA head over one (question, context) pair and return the best
    answer span with its score.

    SQuAD 2.0 trick: the span that starts AND ends on the [CLS] token is the
    model's "no answer" prediction. delta = best_span - no_answer  >  0
    means the model believes the context really contains the answer.
    """
    enc = tokenizer(question, context, return_tensors="pt",
                    truncation=True, max_length=512)
    enc_dev = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc_dev)

    start_logits = out.start_logits[0]
    end_logits = out.end_logits[0]

    # sequence_ids(): None = special token, 0 = question, 1 = context.
    # Only tokens of the context may start or end an answer.
    seq_ids = enc.sequence_ids()
    ctx = [i for i, s in enumerate(seq_ids) if s == 1]

    null_score = (start_logits[0] + end_logits[0]).item()   # [CLS] = "no answer"

    top_starts = torch.topk(start_logits[ctx], k=min(n_best, len(ctx))).indices.tolist()
    top_ends = torch.topk(end_logits[ctx], k=min(n_best, len(ctx))).indices.tolist()

    spans = []
    for si in top_starts:
        for ei in top_ends:
            if ei < si or ei - si > max_span_len:
                continue
            score = (start_logits[ctx[si]] + end_logits[ctx[ei]]).item()
            spans.append((score, si, ei))

    if not spans:  # degenerate context -> fall back to the single best token
        si = int(torch.argmax(start_logits[ctx]).item())
        spans = [((start_logits[ctx[si]] + end_logits[ctx[si]]).item(), si, si)]

    spans.sort(key=lambda x: x[0], reverse=True)
    best_score, s_local, e_local = spans[0]
    s_glob, e_glob = ctx[s_local], ctx[e_local]

    answer_text = tokenizer.decode(enc["input_ids"][0][s_glob:e_glob + 1]).strip()
    delta = best_score - null_score
    confidence = 1.0 / (1.0 + math.exp(-max(min(delta, 50.0), -50.0)))  # sigmoid

    return {"text": answer_text, "delta": delta, "confidence": confidence}

---
# Part 7: The Question-Answering Tool



In [ ]:
NO_ANSWER_MARGIN = 0.0   # the best span must beat "no answer" by this much
TOP_K_CHUNKS = 8         # chunks retrieved per question


def answer_question(question: str, top_k: int = TOP_K_CHUNKS) -> dict:
    """
    The full QA tool: retrieve the most relevant chunks, run the QA model on
    each, and return the best answer span — or None if the document does
    not contain the answer.
    """
    hits = retriever.top_k(question, k=top_k)
    if not hits:
        return {"question": question, "answer": None, "confidence": 0.0,
                "page": None, "evidence": None}

    best = None
    for chunk, sim in hits:
        cand = extract_answer(question, chunk["text"])
        cand["chunk"] = chunk
        if best is None or cand["delta"] > best["delta"]:
            best = cand

    if best["delta"] <= NO_ANSWER_MARGIN or not best["text"]:
        return {"question": question, "answer": None,
                "confidence": round(best["confidence"], 3),
                "page": None, "evidence": None}

    # build a short evidence snippet around the answer span
    chunk_text = best["chunk"]["text"]
    pos = chunk_text.lower().find(best["text"].lower())
    if pos >= 0:
        lo = max(0, pos - 90)
        hi = min(len(chunk_text), pos + len(best["text"]) + 90)
        evidence = ("..." if lo else "") + chunk_text[lo:hi].replace("\n", " ").strip()
        if hi < len(chunk_text):
            evidence += "..."
    else:
        evidence = chunk_text[:200].replace("\n", " ")

    return {"question": question, "answer": best["text"],
            "confidence": round(best["confidence"], 3),
            "page": best["chunk"]["page"], "evidence": evidence}


def ask(question: str) -> dict:
    """Ask the tool a question and pretty-print the result."""
    r = answer_question(question)
    print(f"Q: {question}")
    if r["answer"] is None:
        print("A: I could not find the answer to this question in the document.")
    else:
        print(f"A: {r['answer']}   (confidence: {r['confidence']:.2f}, source: page {r['page']})")
        print(textwrap.fill(r["evidence"], width=100,
                            initial_indent="   evidence: ",
                            subsequent_indent="             "))
    print("-" * 78)
    return r


print("QA tool ready.")

QA tool ready.


---
# Part 8: Test — the Assignment Questions



In [ ]:
TEST_SET = [
    ("What is the capital city of Bangladesh?", ["Dhaka"]),
    ("Which river is considered the lifeline of Bangladesh?", ["Padma", "Padma River"]),
    ("In which year did Bangladesh gain independence?", ["1971"]),
    ("What is the official language of Bangladesh?", ["Bengali", "Bangla"]),
    ("What is the name of the world's largest mangrove forest located in Bangladesh?", ["Sundarbans", "Sundarban"]),
    ("Which currency is used in Bangladesh?", ["Taka", "Bangladeshi Taka"]),
    ("What is the national animal of Bangladesh?", ["Royal Bengal Tiger", "Bengal Tiger"]),
    ("Which ancient fortified urban site in northern Bangladesh was historically known as Pundranagara?",
     ["Mahasthangarh", "Mahasthan", "Mahasthangarh (Pundranagara)"]),
    ("Which archaeological site in Narsingdi yielded artifacts demonstrating ancient maritime trade networks?",
     ["Wari-Bateshwar", "Wari Bateshwar"]),
    ("What key crop drove the expansion of settled agriculture and forest clearing in early Bengal?",
     ["Rice", "Wet rice", "Wet-rice", "Paddy"]),
    ("What term describes the moving boundary where wilderness was cleared for agricultural cultivation in historical Bengal?",
     ["Agrarian frontier", "Agrarian frontiers"]),
    ("What are the unstable, temporary sandbars and islands created by river erosion and sedimentation called?",
     ["Chars", "Char", "Char lands"]),
    ("Which mangrove forest ecosystem is situated in the active delta along the Bay of Bengal?",
     ["Sundarbans", "Sundarban"]),
    ("Which three river systems join together to deposit silt and form the Bengal delta?",
     ["Ganges, Brahmaputra, and Meghna", "Ganges, Brahmaputra, Meghna", "Padma, Jamuna, Meghna"]),
]


results = []
print(f"Asking {len(TEST_SET)} questions about: {PDF_PATH}\n")
for question, _gold in TEST_SET:
    results.append(ask(question))

Asking 14 questions about: A-History-of-Bangladesh.pdf

Q: What is the capital city of Bangladesh?
A: Dhaka   (confidence: 1.00, source: page 27)
   evidence: ...ch used to be written as ‘Dacca’ in English-language texts, took its more correct
             form of Dhaka (D˙ ha¯ka¯) in the 1980s. Similarly, ‘Calcutta’ became Kolkata
             (Kalka¯ta¯) in 2001. Rathe...
------------------------------------------------------------------------------
Q: Which river is considered the lifeline of Bangladesh?
A: Ganges   (confidence: 0.71, source: page 67)
   evidence: east, Chryse. There is a river near it called the Ganges, and it rises and falls in the
             same way as the Nile. On its bank is a market-town which h...
------------------------------------------------------------------------------
Q: In which year did Bangladesh gain independence?
A: 1971   (confidence: 0.94, source: page 217)
   evidence: ...historical understanding. In Bangladesh it has been resolutely displac

---
# Part 9: Robustness — Questions the Document Cannot Answer




In [ ]:
OUT_OF_DOC_QUESTIONS = [
    "Who is the current prime minister of Japan?",
    "What is the population of Brazil?",
    "How many universities are there in Dhaka?",
]

for question in OUT_OF_DOC_QUESTIONS:
    ask(question)

Q: Who is the current prime minister of Japan?
A: I could not find the answer to this question in the document.
------------------------------------------------------------------------------
Q: What is the population of Brazil?
A: I could not find the answer to this question in the document.
------------------------------------------------------------------------------
Q: How many universities are there in Dhaka?
A: four   (confidence: 0.82, source: page 184)
   evidence: and four new universities were added to the colonial- era University of Dhaka: the
             University of R...
------------------------------------------------------------------------------


---
# Part 10: Evaluation — Exact Match and F1



In [ ]:
def normalize_answer(s: str) -> str:
    """SQuAD-style normalization: lowercase, remove articles and punctuation."""
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return " ".join(s.split())


def exact_match(pred: str, gold: str) -> float:
    return float(normalize_answer(pred) == normalize_answer(gold))


def f1_score(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    common = set(pred_tokens) & set(gold_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


rows = []
for (question, golds), r in zip(TEST_SET, results):
    pred = r["answer"] or ""
    rows.append({
        "Question": question,
        "Gold answer": " / ".join(golds),
        "Predicted": pred if pred else "-",
        "EM": max(exact_match(pred, g) for g in golds),
        "F1": max(f1_score(pred, g) for g in golds),
        "Confidence": r["confidence"],
        "Page": r["page"],
    })

eval_df = pd.DataFrame(rows)
display(eval_df.style
        .format({"EM": "{:.0f}", "F1": "{:.3f}",
                 "Confidence": "{:.2f}", "Page": "{:.0f}"}, na_rep="-")
        .hide(axis="index"))

print(f"\nExact Match: {eval_df['EM'].mean() * 100:.1f}%   |   F1: {eval_df['F1'].mean() * 100:.1f}%")

Question,Gold answer,Predicted,EM,F1,Confidence,Page
What is the capital city of Bangladesh?,Dhaka,Dhaka,1,1.000,1.00,27
Which river is considered the lifeline of Bangladesh?,Padma / Padma River,Ganges,0,0.000,0.71,67
In which year did Bangladesh gain independence?,1971,1971,1,1.000,0.94,217
What is the official language of Bangladesh?,Bengali / Bangla,Bengali,1,1.000,1.00,27
What is the name of the world's largest mangrove forest located in Bangladesh?,Sundarbans / Sundarban,-,0,0.000,0.46,-
Which currency is used in Bangladesh?,Taka / Bangladeshi Taka,taka,1,1.000,1.00,206
What is the national animal of Bangladesh?,Royal Bengal Tiger / Bengal Tiger,Bengal tiger,1,1.000,1.00,277
Which ancient fortified urban site in northern Bangladesh was historically known as Pundranagara?,Mahasthangarh / Mahasthan / Mahasthangarh (Pundranagara),Mahasthan,1,1.000,0.99,43
Which archaeological site in Narsingdi yielded artifacts demonstrating ancient maritime trade networks?,Wari-Bateshwar / Wari Bateshwar,Wari-Bateshwar,1,1.000,0.98,42
What key crop drove the expansion of settled agriculture and forest clearing in early Bengal?,Rice / Wet rice / Wet-rice / Paddy,wet-rice,1,1.000,0.95,58



Exact Match: 71.4%   |   F1: 77.6%


---
# Part 11: Ask random questions

 Type exit to stop.


In [ ]:
try:
    while True:
        user_question = input("Ask a question about the document (type 'exit' to stop): ").strip()
        if user_question.lower() in {"exit", "quit"}:
            break
        if user_question:
            ask(user_question)
except EOFError:
    pass